# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the **FAIR²** dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema, accessible via the following URL:

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and examine high-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Initialize the dataset
dataset = mlc.Dataset(croissant_url)

# Show summary information from the metadata
metadata = dataset.metadata
print(f"Dataset Title: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Cite As: {getattr(metadata, 'citeAs', '')}")
print(f"Data Collection Notes: {getattr(metadata, 'dataCollection', '')}")

# Authors and keywords
authors = getattr(metadata, 'author', [])
if authors:
    print(f"Authors: {[a['@id'] if isinstance(a, dict) and '@id' in a else str(a) for a in authors]}")
keywords = getattr(metadata, 'keywords', [])
if keywords:
    print("Keywords:", ", ".join(keywords))

## 2. Data Overview
Review available **record sets**, their fields and column IDs. All references are by their Croissant `@id`, following best practice for FAIR data exploration.

Here we list all `@id` references for record sets, their fields, and columns to understand the dataset structure.

In [ ]:
# Display all record sets and their fields/columns by @id

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets declared at the top level. Attempting to discover via underlying dataset structure.")

# mlcroissant auto-discovers available record_set @ids this way:
record_set_ids = []
for rs in dataset.record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        # Each field is either an @id string or a dict with @id
        fid = f if isinstance(f, str) else f.get('@id', 'UNKNOWN')
        print(f"    - Field @id: {fid}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for c in columns:
            cid = c if isinstance(c, str) else c.get('@id', 'UNKNOWN')
            print(f"    - Column @id: {cid}")
    print()

if not record_set_ids:
    print("No record sets discovered. Please check the Croissant definition.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references (record sets, fields/columns) are by `@id` as explored above.

_Below, we load all data for the main record set by its `@id`._

In [ ]:
# Choose the main RecordSet @id to extract data
assert len(record_set_ids) > 0, "No record sets were found in metadata."
# For this dataset, we select the first RecordSet as an example (or replace with desired @id)
main_record_set_id = record_set_ids[0]
print(f"Loading data from RecordSet: {main_record_set_id}")

# Extract records from the RecordSet
records_iter = dataset.records(record_set=main_record_set_id)
records = list(records_iter)

df = pd.DataFrame(records)
print(f"Loaded DataFrame columns (fields by @id):\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing, or grouping data. All access is by `@id`.

#### Example: Filtering by 'cr:coveragePercentage' field, Normalization, and Grouping

- Select a numeric field, such as `cr:coveragePercentage` or similar.
- Apply a threshold, normalization, and group statistics.
- _Replace `cr:coveragePercentage` and grouping field as appropriate with valid @id names discovered in Section 2._

In [ ]:
# Specify the field @id for a numeric field (e.g., coverage %, or molecular weight in kDa)
numeric_field = None
candidate_numeric_fields = [col for col in df.columns if (
    ('coverage' in col.lower()) or ('kda' in col.lower()) or (df[col].dtype in ['int64', 'float64'])
)]

if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]  # pick the first matching field
else:
    # fallback: pick the first float/integer column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

if numeric_field is None:
    raise ValueError("Could not find a numeric field in the record set!")

print(f"Analyzing numeric field (by @id): {numeric_field}")

# Drop NA and filter for values greater than an example threshold (e.g., 10)
threshold = 10
filtered_df = df[df[numeric_field].fillna(0) > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
print(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a categorical or annotation field, e.g., 'cr:sample' or 'cr:proteinType'
group_field_candidates = [c for c in df.columns if ('type' in c.lower()) or ('sample' in c.lower())]
group_field = group_field_candidates[0] if group_field_candidates else None

if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields by their `@id`. For example, display a histogram of the selected numeric field and, if a group field exists, a boxplot by group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Histogram of the normalized numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True)
plt.title(f"Distribution of Normalized {numeric_field} (@id)")
plt.xlabel(f"{numeric_field} (z-score)")
plt.ylabel("Frequency")
plt.show()

# If grouping field is available, draw boxplot
if group_field is not None and group_field in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=filtered_df,
        x=group_field,
        y=numeric_field
    )
    plt.title(f"{numeric_field} by {group_field} (by @id)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 6. Conclusion
- We used [`mlcroissant`](https://github.com/mlcommons/croissant) to explore the FAIR² dataset via its [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).
- Data structure was explored using `@id` references for all record sets and fields.
- Basic EDA (filtering, normalization, grouping) and visualizations were performed using only standard Python and Pandas/Seaborn tools.

This notebook may be extended to include more domain-specific feature engineering or downstream machine learning workflows. For best practices, always reference fields, record sets, and columns by their Croissant `@id`! If you use these results, please cite the dataset as indicated above.